# Ultimate GTF Strategy Backtester — Google Colab

Backtests the **GTF supply/demand zone strategy** on NSE equities using Fyers historical data,
entirely in the browser. No local Python install needed.

---

### ⚠️ Disclaimer

> *"Investment in securities market are subject to market risks. Read all the related documents carefully before investing."*

This notebook is for **educational and research purposes only**. It is not a recommendation to buy or sell
any financial instrument. Backtest results are based on historical data and **do not guarantee future
performance**. You are responsible for your own Fyers API credentials and for complying with Fyers' terms
of service. Automated trading carries significant risk — always test in a paper-trading environment first.

---

### How to use

Run the cells **top to bottom**. Each step is a numbered section:

| Step | What it does |
| --- | --- |
| 1 | Install dependencies |
| 2 | Pull the strategy code from GitHub |
| 3 | Log in to Fyers and save an access token |
| 4 | Choose which stocks to test |
| 5 | Set the backtest parameters |
| 6 | Run the backtest |
| 7 | Download the Excel report and view the charts |

**You will need:** a Fyers API account with an app created (Client ID + Secret Key), and the app's
redirect URI set to `http://127.0.0.1`.

> **Note on runtimes:** Colab wipes the VM when the runtime disconnects. If that happens, re-run from
> Step 1 — including the token step, since `fyers_access_token.txt` is stored on the VM.

---
## Step 1 — Install dependencies

`pandas` and `numpy` ship with Colab; this adds the Fyers client, the Excel writer and the timezone data.
Takes about 20 seconds.

In [ ]:
#@title Install packages { display-mode: "form" }
!pip install -q fyers-apiv3 openpyxl pytz tabulate

import pandas as pd, numpy as np
print("pandas", pd.__version__, "| numpy", np.__version__)

import fyers_apiv3
print("fyers-apiv3 installed \u2713")

---
## Step 2 — Get the strategy code

Clones the repo into `/content/` and imports `gtf_strategy_automation` as `gtf`. Re-running this cell
pulls the latest changes.

The strategy logic lives in the repo, not in this notebook — so any fix pushed to GitHub shows up here
on the next run.

In [ ]:
#@title Clone the repository and import the strategy { display-mode: "form" }
import os, sys, subprocess, importlib

REPO_URL = "https://github.com/ersid111/Stocks.git"  #@param {type:"string"}
BRANCH   = "claude/strategy-document-review-kxjk0s"  #@param {type:"string"}

CLONE_DIR = "/content/Stocks"
WORK_DIR  = os.path.join(CLONE_DIR, "gtf_strategy")

if os.path.isdir(os.path.join(CLONE_DIR, ".git")):
    print("Repo already present \u2014 pulling latest...")
    subprocess.run(["git", "-C", CLONE_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", CLONE_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", CLONE_DIR, "pull", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1", REPO_URL, CLONE_DIR],
                   check=True)

# Work inside the strategy folder so the token file and reports land together
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)
if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)

import gtf_strategy_automation as gtf
importlib.reload(gtf)   # pick up changes if the cell is re-run

print(f"\nWorking directory: {os.getcwd()}")
print(f"Strategy module loaded \u2713  (HTF={gtf.HTF_INTERVAL}, LTF={gtf.LTF_INTERVAL})")

---
## Step 3 — Fyers login

Colab runs on a remote machine with no browser, so the desktop `generate_token.py` flow doesn't apply
here. Instead:

1. **Cell 3a** prints a login link.
2. Open it in a **new tab**, log in to Fyers.
3. You'll be redirected to `http://127.0.0.1/?...auth_code=...` and the page will fail to load
   (**this is normal and expected** — nothing is listening on that address).
4. Copy the **entire URL** from the address bar.
5. Paste it into **cell 3b**.

The token is saved to `fyers_access_token.txt` on the Colab VM and lasts for the trading day.

> **Already have a valid token?** Skip 3a/3b and use cell **3c** to paste it directly.

In [ ]:
#@title 3a. Enter credentials and generate the login link { display-mode: "form" }
from fyers_apiv3 import fyersModel
from IPython.display import display, HTML

CLIENT_ID    = "AAAA1AAAAA-999"     #@param {type:"string"}
SECRET_KEY   = "1234567891"         #@param {type:"string"}
REDIRECT_URI = "http://127.0.0.1"   #@param {type:"string"}

# Tell the strategy module which credentials and token file to use
gtf.CLIENT_ID  = CLIENT_ID
gtf.TOKEN_FILE = os.path.join(WORK_DIR, "fyers_access_token.txt")

session = fyersModel.SessionModel(
    client_id=CLIENT_ID,
    secret_key=SECRET_KEY,
    redirect_uri=REDIRECT_URI,
    response_type="code",
    grant_type="authorization_code",
)

auth_url = session.generate_authcode()

display(HTML(
    f'<div style="font-family:system-ui,sans-serif;line-height:1.6">'
    f'<p><b>1.</b> Open this link in a new tab and log in to Fyers:</p>'
    f'<p><a href="{auth_url}" target="_blank" rel="noopener" '
    f'style="font-size:15px;font-weight:600">\u2192 Log in to Fyers</a></p>'
    f'<p><b>2.</b> You will land on a <i>"site can\u2019t be reached"</i> page at '
    f'<code>127.0.0.1</code>. That is expected.</p>'
    f'<p><b>3.</b> Copy the <b>full URL</b> from the address bar and paste it into the next cell.</p>'
    f'</div>'
))

In [ ]:
#@title 3b. Paste the redirect URL to create the token { display-mode: "form" }
redirect_url = input("Paste the full redirect URL (starts with http://127.0.0.1): ").strip()

try:
    auth_code = redirect_url.split("auth_code=")[1].split("&")[0]
except (IndexError, AttributeError):
    raise SystemExit("Could not find 'auth_code=' in that URL. Paste the FULL URL from the address bar.")

print(f"Extracted auth_code: {auth_code[:10]}...")

session.set_token(auth_code)
response = session.generate_token()

if "access_token" in response:
    with open(gtf.TOKEN_FILE, "w") as f:
        f.write(response["access_token"])
    print(f"\nSUCCESS \u2713  Token saved to {gtf.TOKEN_FILE}")
else:
    print("\nFAILED TO GENERATE TOKEN")
    print("Error:", response.get("message", "Unknown error"))
    print("The auth code expires within a minute \u2014 re-run cell 3a and log in quickly.")

In [ ]:
#@title 3c. (Alternative) Paste an access token you already have { display-mode: "form" }
EXISTING_TOKEN = ""  #@param {type:"string"}

if EXISTING_TOKEN.strip():
    gtf.TOKEN_FILE = os.path.join(WORK_DIR, "fyers_access_token.txt")
    with open(gtf.TOKEN_FILE, "w") as f:
        f.write(EXISTING_TOKEN.strip())
    print(f"Token saved to {gtf.TOKEN_FILE} \u2713")
else:
    print("No token pasted \u2014 skipping this cell (use 3a/3b instead).")

In [ ]:
#@title 3d. Verify the connection { display-mode: "form" }
fyers_client = gtf.initialize_fyers()

if fyers_client is None:
    print("\nLogin failed \u2014 fix the above before continuing to Step 4.")
else:
    print("Ready to backtest \u2713")

---
## Step 4 — Choose your stocks

One **NSE symbol per line**, plain symbols with no `.NS` suffix and no exchange prefix — the script turns
`HDFCBANK` into `NSE:HDFCBANK-EQ` for you. Lines starting with `#` are ignored.

In [ ]:
#@title Set the stock list { display-mode: "form" }
SYMBOLS = "HDFCBANK, RELIANCE, SBIN, INFY, TCS"  #@param {type:"string"}

tickers = [s.strip().upper() for s in SYMBOLS.replace("\n", ",").split(",")
           if s.strip() and not s.strip().startswith("#")]

with open(os.path.join(WORK_DIR, "stock_list.txt"), "w") as f:
    f.write("\n".join(tickers) + "\n")

print(f"{len(tickers)} symbols queued:")
print(", ".join(tickers))

---
## Step 5 — Backtest parameters

These overwrite the defaults in the strategy module for this session only — the file on disk is untouched.

**Watch the warmup.** The backtest loop only starts after `EMA_TP_PERIOD + ATR_LOOKBACK + 2` bars
(216 by default). At 15m that's roughly 9 trading days of data consumed before the first possible trade,
so give yourself a date range comfortably longer than that or you'll get zero trades.

Fyers also caps how far back intraday history goes — if a symbol returns no candles, shorten the range.

In [ ]:
#@title Set parameters { display-mode: "form" }
#@markdown ### Date range
START_DATE = "2025-09-01"  #@param {type:"date"}
END_DATE   = "2025-09-30"  #@param {type:"date"}

#@markdown ### Timeframes
HTF_INTERVAL = "60m"  #@param ["5m", "15m", "30m", "60m", "1d"]
LTF_INTERVAL = "15m"  #@param ["5m", "15m", "30m", "60m", "1d"]

#@markdown ### Capital and risk
INITIAL_CAPITAL = 100000  #@param {type:"number"}
RISK_PER_TRADE  = 1000    #@param {type:"number"}
MAX_R_MULTIPLE  = 3.0     #@param {type:"number"}

#@markdown ### Zone and filter settings
MAX_BASE_CANDLES            = 3     #@param {type:"integer"}
BASE_CANDLE_RATIO           = 0.33  #@param {type:"number"}
MIN_TRADE_SCORE             = 4.5   #@param {type:"number"}
MIN_WICK_TO_RANGE_RATIO     = 0.4   #@param {type:"number"}
MAX_ZONE_WIDTH_ATR_MULTIPLE = 1.5   #@param {type:"number"}
ATR_LOOKBACK                = 14    #@param {type:"integer"}
EMA_TP_PERIOD               = 200   #@param {type:"integer"}

#@markdown ### Pacing
#@markdown Seconds to wait between symbols, to stay inside Fyers' rate limits.
API_DELAY_SECONDS = 1.0  #@param {type:"number"}

from datetime import time as _time

gtf.START_DATE      = START_DATE
gtf.END_DATE        = END_DATE
gtf.HTF_INTERVAL    = HTF_INTERVAL
gtf.LTF_INTERVAL    = LTF_INTERVAL
gtf.INITIAL_CAPITAL = float(INITIAL_CAPITAL)
gtf.RISK_PER_TRADE  = float(RISK_PER_TRADE)
gtf.MAX_R_MULTIPLE  = float(MAX_R_MULTIPLE)
gtf.MAX_BASE_CANDLES            = int(MAX_BASE_CANDLES)
gtf.BASE_CANDLE_RATIO           = float(BASE_CANDLE_RATIO)
gtf.MIN_TRADE_SCORE             = float(MIN_TRADE_SCORE)
gtf.MIN_WICK_TO_RANGE_RATIO     = float(MIN_WICK_TO_RANGE_RATIO)
gtf.MAX_ZONE_WIDTH_ATR_MULTIPLE = float(MAX_ZONE_WIDTH_ATR_MULTIPLE)
gtf.ATR_LOOKBACK                = int(ATR_LOOKBACK)
gtf.EMA_TP_PERIOD               = int(EMA_TP_PERIOD)

# The intraday rule set follows the entry timeframe
gtf.IS_INTRADAY_STRATEGY = LTF_INTERVAL in ["5m", "15m", "30m", "60m"]

warmup = gtf.EMA_TP_PERIOD + gtf.ATR_LOOKBACK + 2
print(f"Window        : {gtf.START_DATE} \u2192 {gtf.END_DATE}")
print(f"Timeframes    : HTF {gtf.HTF_INTERVAL} / LTF {gtf.LTF_INTERVAL}")
print(f"Capital       : Rs {gtf.INITIAL_CAPITAL:,.0f} | risk Rs {gtf.RISK_PER_TRADE:,.0f}/trade "
      f"| target {gtf.MAX_R_MULTIPLE}R")
print(f"Intraday rules: {'ON  (14:45 IST cutoff, no overnight shorts)' if gtf.IS_INTRADAY_STRATEGY else 'OFF (delivery)'}")
print(f"Warmup        : {warmup} bars consumed before the first possible trade")

---
## Step 6 — Run the backtest

Loops over every symbol. Each one needs two API calls (HTF + LTF candles), so a long list takes a while.

The results accumulate in module-level lists, which this cell **clears first** — so you can safely
re-run it after changing parameters without double-counting the previous run's trades.

In [ ]:
#@title Run { display-mode: "form" }
import time as _t

# Clear results from any previous run in this session
gtf.ALL_SUMMARY_RESULTS.clear()
gtf.ALL_COMPLETED_TRADES.clear()

failed = []

for n, ticker in enumerate(tickers, 1):
    print(f"\n[{n}/{len(tickers)}] {ticker}")
    print("-" * 60)
    try:
        result = gtf.run_backtest(ticker, gtf.START_DATE, gtf.END_DATE)
        if result:
            gtf.ALL_SUMMARY_RESULTS.append(result)
        else:
            failed.append(ticker)
    except Exception as e:
        print(f"  ERROR on {ticker}: {type(e).__name__}: {e}")
        failed.append(ticker)

    if n < len(tickers):
        _t.sleep(API_DELAY_SECONDS)

print("\n" + "=" * 60)
print(f"Finished: {len(gtf.ALL_SUMMARY_RESULTS)} symbols completed, {len(failed)} skipped")
if failed:
    print("Skipped (no data or error):", ", ".join(failed))

In [ ]:
#@title Results { display-mode: "form" }
from IPython.display import display

summary_df = pd.DataFrame(gtf.ALL_SUMMARY_RESULTS)
trades_df  = pd.DataFrame(gtf.ALL_COMPLETED_TRADES)

if summary_df.empty:
    print("No completed backtests. Check Step 6's output for API errors, and see the "
          "troubleshooting notes at the bottom of this notebook.")
else:
    summary_df = summary_df.sort_values("Total P&L", ascending=False).reset_index(drop=True)

    total_pnl    = summary_df["Total P&L"].sum()
    total_trades = int(summary_df["Total Trades"].sum())
    wins         = int((trades_df["Status"].eq("TP") |
                        (trades_df["Status"].eq("FORCED_EOD_CLOSE") & trades_df["PnL"].gt(0))).sum()) \
                   if not trades_df.empty else 0
    overall_wr   = (wins / total_trades * 100) if total_trades else 0.0
    avg_r        = trades_df["R_Achieved"].mean() if not trades_df.empty else 0.0

    print("=" * 68)
    print("CONSOLIDATED MULTI-ASSET BACKTEST REPORT")
    print("=" * 68)
    print(f"Portfolio P&L : Rs {total_pnl:,.2f}")
    print(f"Total trades  : {total_trades}   |   Win rate: {overall_wr:.2f}%   |   Avg R: {avg_r:.2f}")
    print(f"Window        : {gtf.START_DATE} \u2192 {gtf.END_DATE}  "
          f"(HTF {gtf.HTF_INTERVAL} / LTF {gtf.LTF_INTERVAL})")
    print("=" * 68)

    display(summary_df.style.format({
        "Win Rate": "{:.2f}%", "Total P&L": "Rs {:,.2f}", "Final Capital": "Rs {:,.2f}",
    }).set_caption("Per-symbol summary"))

    if not trades_df.empty:
        print(f"\nTrade outcomes: {trades_df['Status'].value_counts().to_dict()}")
        display(trades_df.head(25))
    else:
        print("\nNo trades were triggered \u2014 every signal was filtered out. "
              "Try a longer date range or a lower MIN_TRADE_SCORE.")

---
## Step 7 — Export and visualise

The Excel report has two sheets: **Summary_Report** (P&L and win rate per symbol) and
**Detailed_Trades** (every fill, with IST timestamps).

In [ ]:
#@title Download the Excel report { display-mode: "form" }
from datetime import datetime

if summary_df.empty:
    print("Nothing to export yet \u2014 run Step 6 first.")
else:
    report_name = f"Backtest_Report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.xlsx"
    report_path = os.path.join(WORK_DIR, report_name)

    with pd.ExcelWriter(report_path, engine="openpyxl") as writer:
        summary_df.to_excel(writer, sheet_name="Summary_Report", index=False)
        trades_df.to_excel(writer, sheet_name="Detailed_Trades", index=False)

    print(f"Saved: {report_path}")

    try:
        from google.colab import files
        files.download(report_path)
    except Exception as e:
        print(f"(Automatic download unavailable: {e})")
        print("Grab it from the file browser in the left sidebar instead.")

In [ ]:
#@title Charts \u2014 equity curve and P&L by symbol { display-mode: "form" }
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

BLUE, RED = "#2a78d6", "#e34948"      # validated pair: profit / loss
INK, MUTED = "#0b0b0b", "#52514e"

if trades_df.empty:
    print("No trades to plot.")
else:
    plot_df = trades_df.copy()
    plot_df["Exit Date"] = pd.to_datetime(plot_df["Exit Date"])
    plot_df = plot_df.sort_values("Exit Date")
    plot_df["Cumulative P&L"] = plot_df["PnL"].cumsum()

    def _style(ax):
        ax.spines[["top", "right"]].set_visible(False)
        ax.spines[["left", "bottom"]].set_color("#d8d7d2")
        ax.grid(axis="y", color="#eeedea", linewidth=1)
        ax.set_axisbelow(True)
        ax.tick_params(colors=MUTED, labelsize=9, length=0)

    # ── Equity curve: one series, so no legend needed \u2014 the title names it ──
    fig, ax = plt.subplots(figsize=(11, 4.2))
    ax.axhline(0, color="#c9c8c3", linewidth=1)
    ax.plot(plot_df["Exit Date"], plot_df["Cumulative P&L"],
            color=BLUE, linewidth=2, solid_capstyle="round")
    final = plot_df["Cumulative P&L"].iloc[-1]
    ax.scatter([plot_df["Exit Date"].iloc[-1]], [final], color=BLUE, s=42, zorder=3)
    ax.annotate(f"  Rs {final:,.0f}", (plot_df["Exit Date"].iloc[-1], final),
                color=INK, fontsize=10, fontweight="bold", va="center")
    ax.set_title("Cumulative P&L across all trades", color=INK, fontsize=13,
                 fontweight="bold", loc="left", pad=12)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:,.0f}"))
    ax.set_ylabel("Rupees", color=MUTED, fontsize=9)
    _style(ax)
    fig.autofmt_xdate()
    plt.tight_layout(); plt.show()

    # ── P&L by symbol: colour marks the sign, the axis and labels state it ──
    by_sym = summary_df.sort_values("Total P&L")
    fig, ax = plt.subplots(figsize=(11, max(2.4, 0.45 * len(by_sym) + 1.4)))
    colors = [BLUE if v >= 0 else RED for v in by_sym["Total P&L"]]
    ax.barh(by_sym["Ticker"], by_sym["Total P&L"], color=colors, height=0.62)
    ax.axvline(0, color="#c9c8c3", linewidth=1)
    lo, hi = min(0, by_sym["Total P&L"].min()), max(0, by_sym["Total P&L"].max())
    pad = max(hi - lo, 1) * 0.18
    for y, v in zip(by_sym["Ticker"], by_sym["Total P&L"]):
        ax.annotate(f"{v:,.0f}", (v, y), color=INK, fontsize=9, va="center",
                    ha="left" if v >= 0 else "right",
                    xytext=(4 if v >= 0 else -4, 0), textcoords="offset points")
    ax.set_xlim(lo - pad, hi + pad)   # keep zero in frame without wasting half the canvas
    ax.set_title("Net P&L by symbol (rupees)", color=INK, fontsize=13,
                 fontweight="bold", loc="left", pad=12)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:,.0f}"))
    _style(ax)
    ax.grid(axis="y", visible=False)
    ax.grid(axis="x", color="#eeedea", linewidth=1)
    plt.tight_layout(); plt.show()

---
## Troubleshooting

| Symptom | Fix |
| --- | --- |
| `fyers_access_token.txt not found` | Run cell 3b (or 3c). The Colab VM loses it on disconnect — re-run Step 3. |
| `Fyers API Login Failed` | Token expired (they're good for the trading day). Re-run 3a and 3b. |
| Auth code fails immediately | The code expires within about a minute. Re-run 3a and log in quickly. |
| `Interval ... not supported` | Use `5m`, `15m`, `30m`, `60m`, `1d` or `1mo`. |
| `Not enough LTF data bars available` | The range is shorter than the 216-bar warmup. Widen the dates or lower `EMA_TP_PERIOD`. |
| `No candles data` for a symbol | Wrong symbol, or the range is beyond Fyers' intraday history limit. Check the spelling and shorten the range. |
| Backtest completes but zero trades | Normal with strict filters. Try a longer window, a lower `MIN_TRADE_SCORE`, a lower `MIN_WICK_TO_RANGE_RATIO`, or `MAX_ZONE_WIDTH_ATR_MULTIPLE = 999` to disable the width filter. |
| Rate-limit errors | Raise `API_DELAY_SECONDS` in Step 5. |
| Results look doubled | Re-run the Step 6 cell — it clears previous results at the top. |

### Strategy reference

Full explanation of the zones, filters, scoring and entry types is in
[`gtf_strategy/README.md`](https://github.com/ersid111/Stocks/blob/main/gtf_strategy/README.md).